Enriched Datetimes
==============================================================================

Adding support to `splatlog.rich.enrich` for `datetime` classes.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import datetime as dt
import re

import splatlog as slog
from splatlog.lib.text import fmt
from splatlog.rich.enrich import highlighted
from rich.text import Text
from rich.console import Console
from rich.highlighter import ISO8601Highlighter, RegexHighlighter
from rich.theme import Theme
from rich.style import Style

slog.setup(level=slog.INFO, console=True, theme=slog.rich.THEME_ANSI_DARK)

LOG = slog.getLogger(__name__)

C = Console(theme=slog.rich.THEME_ANSI_DARK)
p = C.print
HL = ISO8601Highlighter()

In [3]:
class MyHighlighter(RegexHighlighter):
    """Highlights the ISO8601 date time strings.
    Regex reference: https://www.oreilly.com/library/view/regular-expressions-cookbook/9781449327453/ch04s07.html
    """

    base_style = "iso8601."
    highlights = [
        #
        # Dates
        #
        # Calendar month (e.g. 2008-08). The hyphen is required
        r"^(?P<year>[0-9]{4})-(?P<month>1[0-2]|0[1-9])$",
        # Calendar date w/o hyphens (e.g. 20080830)
        r"^(?P<date>(?P<year>[0-9]{4})(?P<month>1[0-2]|0[1-9])(?P<day>3[01]|0[1-9]|[12][0-9]))$",
        # Ordinal date (e.g. 2008-243). The hyphen is optional
        r"^(?P<date>(?P<year>[0-9]{4})-?(?P<day>36[0-6]|3[0-5][0-9]|[12][0-9]{2}|0[1-9][0-9]|00[1-9]))$",
        #
        # Weeks
        #
        # Week of the year (e.g., 2008-W35). The hyphen is optional
        r"^(?P<date>(?P<year>[0-9]{4})-?W(?P<week>5[0-3]|[1-4][0-9]|0[1-9]))$",
        # Week date (e.g., 2008-W35-6). The hyphens are optional
        r"^(?P<date>(?P<year>[0-9]{4})-?W(?P<week>5[0-3]|[1-4][0-9]|0[1-9])-?(?P<day>[1-7]))$",
        #
        # Times
        #
        # Hours and minutes (e.g., 17:21). The colon is optional
        r"^(?P<time>(?P<hour>2[0-3]|[01][0-9]):?(?P<minute>[0-5][0-9]))$",
        # Hours, minutes, and seconds w/o colons (e.g., 172159)
        r"^(?P<time>(?P<hour>2[0-3]|[01][0-9])(?P<minute>[0-5][0-9])(?P<second>[0-5][0-9]))$",
        # Time zone designator (e.g., Z, +07 or +07:00). The colons and the minutes are optional
        r"^(?P<timezone>(Z|[+-](?:2[0-3]|[01][0-9])(?::?(?:[0-5][0-9]))?))$",
        # Hours, minutes, and seconds with time zone designator (e.g., 17:21:59+07:00).
        # All the colons are optional. The minutes in the time zone designator are also optional
        r"^(?P<time>(?P<hour>2[0-3]|[01][0-9])(?P<minute>[0-5][0-9])(?P<second>[0-5][0-9]))(?P<timezone>Z|[+-](?:2[0-3]|[01][0-9])(?::?(?:[0-5][0-9]))?)$",
        #
        # Date and Time
        #
        # Calendar date with hours, minutes, and seconds (e.g., 2008-08-30 17:21:59 or 20080830 172159).
        # A space is required between the date and the time. The hyphens and colons are optional.
        # This regex matches dates and times that specify some hyphens or colons but omit others.
        # This does not follow ISO 8601
        r"^(?P<date>(?P<year>[0-9]{4})(?P<hyphen>-)?(?P<month>1[0-2]|0[1-9])(?(hyphen)-)(?P<day>3[01]|0[1-9]|[12][0-9])) (?P<time>(?P<hour>2[0-3]|[01][0-9])(?(hyphen):)(?P<minute>[0-5][0-9])(?(hyphen):)(?P<second>[0-5][0-9]))$",
        #
        # XML Schema dates and times
        #
        # Date, with optional time zone (e.g., 2008-08-30 or 2008-08-30+07:00).
        # Hyphens are required. This is the XML Schema 'date' type
        r"^(?P<date>(?P<year>-?(?:[1-9][0-9]*)?[0-9]{4})-(?P<month>1[0-2]|0[1-9])-(?P<day>3[01]|0[1-9]|[12][0-9]))(?P<timezone>Z|[+-](?:2[0-3]|[01][0-9]):[0-5][0-9])?$",
        # Time, with optional fractional seconds and time zone (e.g., 01:45:36 or 01:45:36.123+07:00).
        # There is no limit on the number of digits for the fractional seconds. This is the XML Schema 'time' type
        r"^(?P<time>(?P<hour>2[0-3]|[01][0-9]):(?P<minute>[0-5][0-9]):(?P<second>[0-5][0-9])(?P<frac>\.[0-9]+)?)(?P<timezone>Z|[+-](?:2[0-3]|[01][0-9]):[0-5][0-9])?$",
        # Date and time, with optional fractional seconds and time zone (e.g., 2008-08-30T01:45:36 or 2008-08-30T01:45:36.123Z).
        # This is the XML Schema 'dateTime' type
        r"^(?P<date>(?P<year>-?(?:[1-9][0-9]*)?[0-9]{4})(?P<hyphen>-)(?P<month>1[0-2]|0[1-9])(?(hyphen)-)(?P<day>3[01]|0[1-9]|[12][0-9]))[T\ ](?P<time>(?P<hour>2[0-3]|[01][0-9]):(?P<minute>[0-5][0-9]):(?P<second>[0-5][0-9])(?P<ms>\.[0-9]+)?)(?P<timezone>Z|[+-](?:2[0-3]|[01][0-9]):[0-5][0-9])?$",
    ]

In [4]:
def p_dt(dt_s, styles={}):
    theme = slog.rich.to_theme(styles, base=slog.rich.THEME_ANSI_DARK)
    console = Console(theme=theme)
    console.print(MyHighlighter()(dt_s))

In [5]:
naive = dt.datetime.now()

p_dt(fmt(naive), {"iso8601.hyphen": Style(color="red")})

2026-07-13 00:11:44.136

In [6]:
hl = MyHighlighter()
r = re.compile(hl.highlights[-1])
s = fmt(naive)
for match in r.finditer(s):
    for name in match.groupdict().keys():
        start, end = match.span(name)
        style = f"{MyHighlighter.base_style}{name}"
        LOG.info(
            "group dict match",
            match=match,
            style=style,
            span=(start, end),
            chunk=s[start:end],
        )


INFO        __main__                                                                                               
msg         group dict match                                                                                       
data        match         re.Match        <re.Match object; span=(0, 23), match='2026-07-13 00:11:44.136'>         
            style         str             iso8601.date                                                             
            span          tuple           (0, 10)                                                                  
            chunk         str             2026-07-13                                                      

INFO        __main__                                                                                               
msg         group dict match                                                                                       
data        match         re.Match        <re.Match object; span=(0, 23), match='2026-07-13 00:11:44.136'>         
            style         str             iso8601.year                                                             
            span          tuple           (0, 4)                                                                   
            chunk         str             2026                                                            

INFO        __main__                                                                                               
msg         group dict match                                                                                       
data        match         re.Match        <re.Match object; span=(0, 23), match='2026-07-13 00:11:44.136'>         
            style         str             iso8601.hyphen                                                           
            span          tuple           (4, 5)                                                                   
            chunk         str             -                                                               

INFO        __main__                                                                                               
msg         group dict match                                                                                       
data        match         re.Match        <re.Match object; span=(0, 23), match='2026-07-13 00:11:44.136'>         
            style         str             iso8601.month                                                            
            span          tuple           (5, 7)                                                                   
            chunk         str             07                                                              

INFO        __main__                                                                                               
msg         group dict match                                                                                       
data        match         re.Match        <re.Match object; span=(0, 23), match='2026-07-13 00:11:44.136'>         
            style         str             iso8601.day                                                              
            span          tuple           (8, 10)                                                                  
            chunk         str             13                                                              

...

INFO        __main__                                                                                               
msg         group dict match                                                                                       
data        match         re.Match        <re.Match object; span=(0, 23), match='2026-07-13 00:11:44.136'>         
            style         str             iso8601.hour                                                             
            span          tuple           (11, 13)                                                                 
            chunk         str             00                                                              

INFO        __main__                                                                                               
msg         group dict match                                                                                       
data        match         re.Match        <re.Match object; span=(0, 23), match='2026-07-13 00:11:44.136'>         
            style         str             iso8601.minute                                                           
            span          tuple           (14, 16)                                                                 
            chunk         str             11                                                              

INFO        __main__                                                                                               
msg         group dict match                                                                                       
data        match         re.Match        <re.Match object; span=(0, 23), match='2026-07-13 00:11:44.136'>         
            style         str             iso8601.second                                                           
            span          tuple           (17, 19)                                                                 
            chunk         str             44                                                              

INFO        __main__                                                                                               
msg         group dict match                                                                                       
data        match         re.Match        <re.Match object; span=(0, 23), match='2026-07-13 00:11:44.136'>         
            style         str             iso8601.ms                                                               
            span          tuple           (19, 23)                                                                 
            chunk         str             .136                                                            

INFO        __main__                                                                                               
msg         group dict match                                                                                       
data        match         re.Match        <re.Match object; span=(0, 23), match='2026-07-13 00:11:44.136'>         
            style         str             iso8601.timezone                                                         
            span          tuple           (-1, -1)                                                                 
            chunk         str                                                                             

In [7]:
from splatlog.rich.enrich import enrich_timedelta

td = dt.timedelta(seconds=123456, microseconds=123456)
p(enrich_timedelta(td))

1d 10:17:36.123

In [8]:
slog.rich.capture_riches(enrich_timedelta(td))

'\x1b1\x1b\x1b[2;38;2;99;106;128md\x1b \x1b10\x1b\x1b:\x1b\x1b17\x1b\x1b:\x1b\x1b36\x1b\x1b.\x1b\x1b[2;38;2;198;120;221m123\x1b\n'